# 06 -- Parameter stability

Paired script: `analysis/parameter_stability.py` (real, tested pipeline -- **fixed,
2026-07-22 Codex review finding:** this notebook previously computed its own sweep INLINE
with no CSV/JSON output and no pipeline test, and had a real statistical defect: it compared
`mean_r_diff_when_triggered` across different `giveback_percent` settings, but each setting
triggers on a DIFFERENT subset of paths -- averaging only each setting's own triggered
subset and comparing those means across settings is not a like-for-like comparison).
`parameter_stability.sweep_giveback_percent` now computes the mean over the SAME FULL set of
paths at every setting (a 0.0 contribution when the guard never triggers), so every row is
genuinely comparable to every other row.

**Fixed, 2026-07-22 Codex review finding (third round):** `run()` previously accepted only
caller-created in-memory R paths with no explicit-input CLI or dataset hash -- it now reads
R-paths from a real `r_paths.csv` (documented schema: `path_id, bar_index, r_value`), hashed
via the same `build_report_metadata` every other pipeline uses.

This notebook sweeps a REAL strategy parameter -- the V6.37-style giveback guard's
`giveback_percent` -- across a small grid and reports how the guard's behaviour (trigger
rate, mean R saved/lost over the full path set) changes.

**Uses clearly-labelled SYNTHETIC R-paths.** Real-data run: PENDING.

In [ ]:
import sys
import tempfile
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from analysis.parameter_stability import run

In [ ]:
# Hand-traced against ExitManager.mqh's EM_ShouldGivebackCloseV637 formula --
# the exact same fixture and hand-derivation as tests/test_parameter_stability.py.
#
# Path B = [0.0, 2.0, 1.0, 0.5], actual_final_r = 0.5 (last value):
#   pct=40: trigger_r = 2.0*0.6 = 1.2 -> triggers at index2 (current=1.0<=1.2), r=1.0
#           r_diff = 1.0 - 0.5 = 0.5
#   pct=60: trigger_r = 2.0*0.4 = 0.8 -> triggers at index3 (current=0.5<=0.8), r=0.5
#           r_diff = 0.5 - 0.5 = 0.0
#   pct=80: trigger_r = 2.0*0.2 = 0.4 -> never triggers (0.5 > 0.4 at index3); r_diff = 0.0
#
# Path C = [0.0, 1.25, 1.25], actual_final_r = 1.25:
#   peak reaches exactly the arm threshold but current==trigger_r never holds for
#   any percent > 0 -> never triggers at any swept percent. r_diff = 0.0 always.
PATH_B = [0.0, 2.0, 1.0, 0.5]
PATH_C = [0.0, 1.25, 1.25]

tmp_dir = Path(tempfile.mkdtemp(prefix="themba_paramstability_demo_"))
r_paths_csv = tmp_dir / "r_paths.csv"
rows = []
for path_id, values in {"pB": PATH_B, "pC": PATH_C}.items():
    for bar_index, r_value in enumerate(values):
        rows.append({"path_id": path_id, "bar_index": bar_index, "r_value": r_value})
pd.DataFrame(rows).to_csv(r_paths_csv, index=False)

stability_table = run(
    r_paths_csv,
    [40.0, 60.0, 80.0],
    output_csv=tmp_dir / "stability.csv",
    summary_json=tmp_dir / "summary.json",
    repo_path=PROJECT_ROOT.parents[1],
)
print(stability_table)

by_pct = stability_table.set_index("giveback_percent")
assert by_pct.loc[40.0, "n_triggered"] == 1
assert abs(by_pct.loc[40.0, "mean_r_diff_over_all_paths"] - 0.25) < 1e-9  # (0.5 + 0.0) / 2
assert by_pct.loc[60.0, "n_triggered"] == 1
assert abs(by_pct.loc[60.0, "mean_r_diff_over_all_paths"] - 0.0) < 1e-9
assert by_pct.loc[80.0, "n_triggered"] == 0
assert abs(by_pct.loc[80.0, "mean_r_diff_over_all_paths"] - 0.0) < 1e-9
# Every row's mean is over the SAME full 2-path set -- never a shrinking
# triggered-only subset (the Codex review finding this notebook now fixes).
assert (stability_table["n_paths"] == 2).all()

## Real-data run: PENDING

A meaningful parameter-stability read needs many real R-paths (real trade histories with
real bar-level MFE tracking), which do not exist yet -- see `analyse_giveback.py`'s own
"Real-data run: PENDING" note (notebook 02).